## Generate prompts

In [1]:
%cd ..
%pwd  # should be "llm-adaptation"

C:\Users\micha\OneDrive - Univerzita Karlova\research\2024-LLM-DEECo\llm-adaptation


C:\Users\micha\OneDrive - Univerzita Karlova\research\2024-LLM-DEECo\llm-adaptation\.venv\Lib\site-packages\IPython\core\magics\osm.py:417: UserWarning: This is now an optional IPython functionality, setting dhist requires you to install the `pickleshare` library.
  self.shell.db['dhist'] = compress_dhist(dhist)[-100:]


'C:\\Users\\micha\\OneDrive - Univerzita Karlova\\research\\2024-LLM-DEECo\\llm-adaptation'

In [14]:
!python main.py farm/configs/default.yaml farm/configs/config_no_battery.yaml generated_adaptations/jinja_prompt_generator.yaml DSL/drones.yaml farm/configs/simulation_description_2.yaml

Config:

example: farm
steps: 300
adaptation_params:
  adapt_every: 10
  config: true
  prompt_template_params:
    components:
      Field:
        id: id
        attributes:
          location:
            name: top, bottom, left, right
            description: rectangle
            if: 'lambda component: False'
            hide_in_generate: true
          left:
            name: left
            format: '{:d}'
          top:
            name: top
          right:
            name: right
          bottom:
            name: bottom
          threat_level:
            name: threat level
            format: '{:.2f}'
            description: bird-threat level between 0 and 1
          necessary_drones_for_full_protection:
            name: for full protection
            format: '{:d} drones'
          arriving_drones:
            name: flying to field
            format: 'lambda value: f''{value:d} drone{"s" if value != 1 else ""}'''
            description: number of drones flying towar

In [13]:
!python main.py farm/configs/default.yaml farm/configs/config_no_battery.yaml generated_adaptations/jinja_prompt_generator.yaml DSL/drones.yaml farm/configs/simulation_description_2.yaml farm/configs/strategy.yaml

Config:

example: farm
steps: 300
adaptation_params:
  adapt_every: 10
  config: true
  prompt_template_params:
    components:
      Field:
        id: id
        attributes:
          location:
            name: top, bottom, left, right
            description: rectangle
            if: 'lambda component: False'
            hide_in_generate: true
          left:
            name: left
            format: '{:d}'
          top:
            name: top
          right:
            name: right
          bottom:
            name: bottom
          threat_level:
            name: threat level
            format: '{:.2f}'
            description: bird-threat level between 0 and 1
          necessary_drones_for_full_protection:
            name: for full protection
            format: '{:d} drones'
          arriving_drones:
            name: flying to field
            format: 'lambda value: f''{value:d} drone{"s" if value != 1 else ""}'''
            description: number of drones flying towar

In [12]:
!python main.py farm/configs/default.yaml farm/configs/config_no_battery.yaml generated_adaptations/jinja_prompt_generator.yaml DSL/drones.yaml farm/configs/simulation_description_2.yaml farm/configs/goal_step_by_step.yaml

Config:

example: farm
steps: 300
adaptation_params:
  adapt_every: 10
  config: true
  prompt_template_params:
    components:
      Field:
        id: id
        attributes:
          location:
            name: top, bottom, left, right
            description: rectangle
            if: 'lambda component: False'
            hide_in_generate: true
          left:
            name: left
            format: '{:d}'
          top:
            name: top
          right:
            name: right
          bottom:
            name: bottom
          threat_level:
            name: threat level
            format: '{:.2f}'
            description: bird-threat level between 0 and 1
          necessary_drones_for_full_protection:
            name: for full protection
            format: '{:d} drones'
          arriving_drones:
            name: flying to field
            format: 'lambda value: f''{value:d} drone{"s" if value != 1 else ""}'''
            description: number of drones flying towar

## Run generated

In [6]:
import subprocess
import os
import sys
from pathlib import Path
import textwrap
import pandas as pd

In [3]:
farm_configs = ["farm/configs/default.yaml", "generated_adaptations/configs/generated.yaml", "farm/configs/config_no_battery.yaml"]
#farm_variants = [f"{llm}_{variant}_1" for llm in ("4o", "o3") for variant in ("default", "sd1", "step-by-step", "strategy")]
farm_variants = [f"{llm}_{variant}_{repeat}" for llm in ("4o", "o3") for variant in ("sd2", "sd2_step-by-step", "sd2_strategy") for repeat in range(2, 5)]
# farm_variants = [f"{llm}_{variant}_2_v2" for llm in ("4o", "o3") for variant in ("sd2", "sd2_strategy")]

In [ ]:
# farm_variants = ["o3_step-by-step_1"]

### Prepare configuration files

In [7]:
# prepare configuration files
for variant in farm_variants:
    variant_path = Path(f"generated_adaptations/configs/farm/{variant}.yaml")
    if not variant_path.exists():
        variant_path.write_text(f"""name: {variant}
log_dir.append: /{variant}
adaptation_name: generated_adaptations.farm.{variant}.SmartFarmAdaptation
adaptation_params:
  adapt_every: 10""")


In [8]:
# prepare python files
for variant in farm_variants:
    variant_path = Path(f"generated_adaptations/farm/{variant}.py")
    if not variant_path.exists():
        variant_path.parent.mkdir(parents=True, exist_ok=True)
        variant_path.write_text("")

### Run

In [10]:
def run(configs, repeats=10, start=1):
    print(configs)
    damages = []
    for repeat in range(repeats):
        print(f"  Run #{repeat + start}/{repeats + start - 1}")

        run_args = [sys.executable, "main.py", *configs, "-s", str(repeat + start), "-e", str(repeat + start)]
        # if repeat % 10 == 0:
        #     run_args.append("--animation")

        # disable TF errors
        env = dict(os.environ, TF_CPP_MIN_LOG_LEVEL="3")

        result = subprocess.run(run_args, capture_output=True, env=env)
        stdout = result.stdout.decode("utf-8")
        stderr = result.stderr.decode("utf-8")

        if stderr:
            stderr_lines = stderr.splitlines()
            stderr = '\n'.join(stderr_lines[-5:])
            print(f"    StdErr: {len(stderr_lines)} lines")
            print(textwrap.indent(stderr, "    "))

        try:
            damage = stdout.partition("damage: ")[2].partition("\n")[0]
            print(f"    Damage: {damage}")
            damages.append(int(damage))
        except ValueError:
            pass

    if len(damages) > 0:
        avg_damage = sum(damages) / len(damages)
        print(f"Average damage: {avg_damage:.1f}")
    else:
        avg_damage = None
    if repeats != len(damages):
        print(f"Errors: {repeats - len(damages)}")
    print()

    return avg_damage

In [8]:
results = pd.DataFrame(columns=["variant", "damage"])

for variant in farm_variants:
    configs = farm_configs + [f"generated_adaptations/configs/farm/{variant}.yaml"]
    dmg = run(configs, repeats=1, start=1)
    results.loc[len(results)] = [variant, dmg]

['farm/configs/default.yaml', 'generated_adaptations/configs/generated.yaml', 'farm/configs/config_no_battery.yaml', 'generated_adaptations/configs/farm/4o_sd2_2.yaml']
  Run #1/1
    The following components have not been assigned to a group: Drone_1, Drone_2, Drone_3, Drone_4, Drone_5, Drone_6, Drone_7, Drone_8
    The following components have not been assigned to a group: Drone_1, Drone_2, Drone_3, Drone_4, Drone_5, Drone_6, Drone_7, Drone_8
    The following components have not been assigned to a group: Drone_1, Drone_2, Drone_3, Drone_4, Drone_5, Drone_6, Drone_7, Drone_8
    The following components have not been assigned to a group: Drone_1, Drone_2, Drone_3, Drone_4, Drone_5, Drone_6, Drone_7, Drone_8
    The following components have not been assigned to a group: Drone_1, Drone_2, Drone_3, Drone_4, Drone_5, Drone_6, Drone_7, Drone_8
    Damage: 291
Average damage: 291.0

['farm/configs/default.yaml', 'generated_adaptations/configs/generated.yaml', 'farm/configs/config_no_batt

In [9]:
results

,variant,damage
0,4o_sd2_2,291.0
1,4o_sd2_3,297.0
2,4o_sd2_4,304.0
3,4o_sd2_step-by-step_2,330.0
4,4o_sd2_step-by-step_3,262.0
5,4o_sd2_step-by-step_4,301.0
6,4o_sd2_strategy_2,301.0
7,4o_sd2_strategy_3,301.0
8,4o_sd2_strategy_4,301.0
9,o3_sd2_2,282.0
